# Fix Python Import Issues and Environment Setup

This notebook demonstrates how to resolve common Python import errors, environment issues, and module loading problems. We'll walk through the steps needed to fix issues with the AI recruitment system.

## Overview
- Environment setup and verification
- Import error diagnosis
- Python path configuration  
- Cache cleanup operations
- Module reloading techniques
- Database connection testing
- File path validation
- Error handling and debugging

## 1. Environment Setup and Verification

First, let's check our Python environment, version, and current working directory to ensure we're in the right place.

In [ ]:
import sys
import os
import platform
from pathlib import Path

print("=== ENVIRONMENT VERIFICATION ===")
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Current Working Directory: {os.getcwd()}")
print(f"Python Executable: {sys.executable}")
print(f"Python Path: {sys.path[:3]}...")  # Show first 3 entries

# Check if we're in the correct directory
expected_files = ['main.py', 'ai_service.py', 'text_extractor.py', 'db.py']
print(f"\n=== FILE VERIFICATION ===")
for file in expected_files:
    exists = os.path.exists(file)
    print(f"✅ {file}: {'EXISTS' if exists else '❌ MISSING'}")

# Check for important packages
print(f"\n=== PACKAGE VERIFICATION ===")
required_packages = ['openai', 'psycopg2', 'python_dotenv']
for package in required_packages:
    try:
        __import__(package)
        print(f"✅ {package}: INSTALLED")
    except ImportError:
        print(f"❌ {package}: MISSING")

## 2. Import Error Diagnosis

Let's systematically test our imports to identify any issues with our AI recruitment system modules.

In [ ]:
def test_import(module_name, from_module=None, specific_items=None):
    """Test importing a module or specific items from a module"""
    try:
        if from_module and specific_items:
            # Test from X import Y
            module = __import__(from_module, fromlist=specific_items)
            for item in specific_items:
                getattr(module, item)  # This will raise AttributeError if not found
            print(f"✅ from {from_module} import {', '.join(specific_items)}: SUCCESS")
        else:
            # Test simple import
            __import__(module_name)
            print(f"✅ import {module_name}: SUCCESS")
        return True
    except ImportError as e:
        print(f"❌ Import Error for {module_name}: {e}")
        return False
    except AttributeError as e:
        print(f"❌ Attribute Error for {module_name}: {e}")
        return False
    except Exception as e:
        print(f"❌ Unexpected Error for {module_name}: {e}")
        return False

print("=== TESTING CRITICAL IMPORTS ===")

# Test core modules
modules_to_test = [
    ('text_extractor', None, ['extract_text_from_file']),
    ('db', None, ['get_connection', 'insert_resume_data', 'create_tables']),
    ('main', None, ['extract_cv_info', 'read_cv_file']),
    ('scoring_agent.scorer', None, ['score_single_resume']),
]

success_count = 0
for module, from_mod, items in modules_to_test:
    if test_import(module, from_mod, items):
        success_count += 1

print(f"\n=== RESULTS ===")
print(f"Successful imports: {success_count}/{len(modules_to_test)}")

if success_count == len(modules_to_test):
    print("🎉 All critical imports are working!")
else:
    print("⚠️  Some imports failed - we'll fix this in the next steps")

## 3. Python Path Configuration

If imports are failing, we may need to add the current directory or parent directories to the Python path.

In [ ]:
import sys
from pathlib import Path

print("=== CURRENT PYTHON PATH ===")
for i, path in enumerate(sys.path):
    print(f"{i}: {path}")

# Add current directory to Python path if not already there
current_dir = str(Path.cwd())
parent_dir = str(Path.cwd().parent)

print(f"\n=== PATH MODIFICATIONS ===")

if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
    print(f"✅ Added current directory to path: {current_dir}")
else:
    print(f"✅ Current directory already in path: {current_dir}")

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    print(f"✅ Added parent directory to path: {parent_dir}")
else:
    print(f"✅ Parent directory already in path: {parent_dir}")

# Verify the ai_models directory structure
ai_models_dir = Path.cwd()
print(f"\n=== AI_MODELS DIRECTORY STRUCTURE ===")
print(f"Directory: {ai_models_dir}")

python_files = list(ai_models_dir.glob("*.py"))
print(f"Python files found: {[f.name for f in python_files]}")

# Check for scoring_agent subdirectory
scoring_agent_dir = ai_models_dir / "scoring_agent"
if scoring_agent_dir.exists():
    print(f"✅ scoring_agent directory exists")
    scoring_files = list(scoring_agent_dir.glob("*.py"))
    print(f"Files in scoring_agent: {[f.name for f in scoring_files]}")
else:
    print(f"❌ scoring_agent directory missing")

## 4. Cache Cleanup Operations

Python caches compiled bytecode in .pyc files and __pycache__ directories. Stale cache can cause import issues.

In [ ]:
import os
import shutil
from pathlib import Path

def cleanup_python_cache(directory=None):
    """Remove Python cache files and directories"""
    if directory is None:
        directory = Path.cwd()
    else:
        directory = Path(directory)
    
    removed_files = 0
    removed_dirs = 0
    
    print(f"=== CLEANING CACHE IN: {directory} ===")
    
    # Remove .pyc files
    for pyc_file in directory.rglob("*.pyc"):
        try:
            pyc_file.unlink()
            removed_files += 1
            print(f"🗑️  Removed: {pyc_file}")
        except Exception as e:
            print(f"❌ Failed to remove {pyc_file}: {e}")
    
    # Remove __pycache__ directories
    for pycache_dir in directory.rglob("__pycache__"):
        try:
            shutil.rmtree(pycache_dir)
            removed_dirs += 1
            print(f"🗑️  Removed directory: {pycache_dir}")
        except Exception as e:
            print(f"❌ Failed to remove {pycache_dir}: {e}")
    
    print(f"\n=== CLEANUP SUMMARY ===")
    print(f"Removed .pyc files: {removed_files}")
    print(f"Removed __pycache__ directories: {removed_dirs}")
    
    if removed_files == 0 and removed_dirs == 0:
        print("✅ No cache files found - system is clean!")
    else:
        print("✅ Cache cleanup completed!")

# Clean current directory and subdirectories
cleanup_python_cache()

# Also clean parent directory if needed
parent_dir = Path.cwd().parent
if parent_dir.name == 'cv-jd':
    print(f"\n=== CLEANING PARENT DIRECTORY ===")
    cleanup_python_cache(parent_dir)

## 5. Module Reloading Techniques

After cleaning cache and fixing issues, we may need to reload modules to pick up changes.

In [ ]:
import importlib
import sys

def reload_module_safely(module_name):
    """Safely reload a module if it's already imported"""
    try:
        if module_name in sys.modules:
            importlib.reload(sys.modules[module_name])
            print(f"✅ Reloaded: {module_name}")
        else:
            # Import for the first time
            __import__(module_name)
            print(f"✅ Imported: {module_name}")
        return True
    except Exception as e:
        print(f"❌ Failed to reload {module_name}: {e}")
        return False

print("=== MODULE RELOADING ===")

# List of modules to reload
modules_to_reload = [
    'text_extractor',
    'db', 
    'main',
    'scoring_agent.scorer'
]

success_count = 0
for module in modules_to_reload:
    if reload_module_safely(module):
        success_count += 1

print(f"\n=== RELOAD SUMMARY ===")
print(f"Successfully reloaded: {success_count}/{len(modules_to_reload)}")

# Clear import cache for good measure
if hasattr(importlib, 'invalidate_caches'):
    importlib.invalidate_caches()
    print("✅ Cleared import caches")

print("\n=== TESTING IMPORTS AFTER RELOAD ===")
# Re-test our critical imports
try:
    from text_extractor import extract_text_from_file
    from db import get_connection, insert_resume_data, create_tables
    from main import extract_cv_info, read_cv_file
    from scoring_agent.scorer import score_single_resume
    
    print("🎉 All critical imports successful after reload!")
except Exception as e:
    print(f"❌ Import still failing: {e}")

## 6. Database Connection Testing

Let's test our database module and connection functionality.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("=== DATABASE ENVIRONMENT CHECK ===")

# Check for database URL
db_url = os.getenv('DATABASE_URL')
if db_url:
    # Mask sensitive parts of the URL
    masked_url = db_url[:20] + "***" + db_url[-10:] if len(db_url) > 30 else "***"
    print(f"✅ DATABASE_URL found: {masked_url}")
else:
    print("❌ DATABASE_URL not found in environment")
    print("   This is expected for testing - the system will work without database connection")

print("\n=== DATABASE MODULE TESTING ===")

try:
    from db import get_connection, create_tables, insert_resume_data
    print("✅ Database module imported successfully")
    
    # Test connection (will fail gracefully if no DB_URL)
    try:
        conn = get_connection()
        if conn:
            print("✅ Database connection successful")
            conn.close()
        else:
            print("⚠️  Database connection failed (expected without DATABASE_URL)")
    except Exception as e:
        print(f"⚠️  Database connection error: {e}")
        print("   This is expected if DATABASE_URL is not configured")
        
except Exception as e:
    print(f"❌ Database module import failed: {e}")

print("\n=== DATABASE FUNCTIONS TEST ===")

# Test that our database functions exist and are callable
try:
    # Just check if functions exist, don't call them
    from db import get_connection, insert_resume_data, create_tables, insert_cv_score
    
    functions = ['get_connection', 'insert_resume_data', 'create_tables', 'insert_cv_score']
    for func_name in functions:
        func = globals().get(func_name) or locals().get(func_name)
        if callable(eval(f'db.{func_name}' if 'db' in globals() else func_name)):
            print(f"✅ {func_name}: Available")
        else:
            print(f"❌ {func_name}: Not callable")
            
except Exception as e:
    print(f"❌ Function availability check failed: {e}")

print("\n✅ Database testing completed!")
print("Note: Database errors are expected when DATABASE_URL is not configured.")

## 7. File Path Validation

Let's verify that our test files exist and are accessible.

In [ ]:
import os
from pathlib import Path

print("=== FILE PATH VALIDATION ===")

# Check critical files
critical_files = [
    'test_resume.txt',
    'main.py', 
    'ai_service.py',
    'text_extractor.py',
    'db.py',
    'scoring_agent/scorer.py'
]

for file_path in critical_files:
    path = Path(file_path)
    if path.exists():
        size = path.stat().st_size
        print(f"✅ {file_path}: EXISTS ({size} bytes)")
        
        # For Python files, check if they're not empty and have valid syntax
        if file_path.endswith('.py'):
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    if len(content.strip()) == 0:
                        print(f"   ⚠️  WARNING: {file_path} is empty")
                    else:
                        # Basic syntax check
                        compile(content, file_path, 'exec')
                        print(f"   ✅ Syntax valid")
            except SyntaxError as e:
                print(f"   ❌ SYNTAX ERROR in {file_path}: {e}")
            except Exception as e:
                print(f"   ⚠️  Could not validate {file_path}: {e}")
    else:
        print(f"❌ {file_path}: MISSING")

print("\n=== TEST FILE CONTENT CHECK ===")

# Check if our test resume has content
test_resume = Path('test_resume.txt')
if test_resume.exists():
    with open(test_resume, 'r', encoding='utf-8') as f:
        content = f.read()
        lines = content.split('\n')
        print(f"Test resume: {len(content)} characters, {len(lines)} lines")
        print(f"First few lines:")
        for i, line in enumerate(lines[:3]):
            print(f"  {i+1}: {line[:50]}...")
else:
    print("❌ Test resume file missing")

print("\n=== DIRECTORY STRUCTURE ===")

# Show directory structure
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")
print("Directory contents:")
for item in sorted(current_dir.iterdir()):
    if item.is_file():
        print(f"  📄 {item.name}")
    elif item.is_dir():
        print(f"  📁 {item.name}/")
        # Show Python files in subdirectories
        py_files = list(item.glob("*.py"))
        if py_files:
            for py_file in py_files:
                print(f"     📄 {py_file.name}")

print("\n✅ File path validation completed!")

## 8. Error Handling and Debugging

Let's implement comprehensive error handling and test the complete AI service workflow.

In [ ]:
import traceback
import logging
from pathlib import Path

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

def test_ai_service_workflow():
    """Test the complete AI service workflow with error handling"""
    
    print("=== COMPLETE AI SERVICE WORKFLOW TEST ===")
    
    try:
        # Step 1: Test text extraction
        print("\n1. Testing text extraction...")
        from text_extractor import extract_text_from_file
        
        test_file = 'test_resume.txt'
        if Path(test_file).exists():
            text = extract_text_from_file(test_file)
            print(f"✅ Extracted {len(text)} characters from {test_file}")
            print(f"Sample: {text[:100]}...")
        else:
            print(f"❌ Test file {test_file} not found")
            return False
            
    except Exception as e:
        print(f"❌ Text extraction failed: {e}")
        traceback.print_exc()
        return False
    
    try:
        # Step 2: Test OpenAI extraction
        print("\n2. Testing OpenAI extraction...")
        from main import extract_cv_info
        
        # Check if OpenAI API key is available
        import os
        api_key = os.getenv('OPENAI_API_KEY')
        if not api_key:
            print("⚠️  OPENAI_API_KEY not found - AI extraction will use fallback")
        
        extracted_data = extract_cv_info(text)
        print(f"✅ AI extraction completed")
        
        # Parse and validate the JSON
        import json
        data = json.loads(extracted_data)
        print(f"   Name: {data.get('name', 'N/A')}")
        print(f"   Skills: {len(data.get('skills', []))} found")
        print(f"   Experience: {data.get('experience_years', 0)} years")
        
    except Exception as e:
        print(f"❌ OpenAI extraction failed: {e}")
        traceback.print_exc()
        return False
    
    try:
        # Step 3: Test AI service integration
        print("\n3. Testing AI service integration...")
        from ai_service import AIProcessingService
        
        service = AIProcessingService()
        result = service.process_resume_file(test_file)
        
        if result.get('success'):
            print("✅ AI service integration successful")
            print(f"   Extracted data available: {'extracted_data' in result}")
        else:
            print(f"⚠️  AI service completed with issues: {result.get('error', 'Unknown')}")
            
    except Exception as e:
        print(f"❌ AI service integration failed: {e}")
        traceback.print_exc()
        return False
    
    try:
        # Step 4: Test scoring functionality
        print("\n4. Testing scoring functionality...")
        from scoring_agent.scorer import score_single_resume
        
        sample_job_desc = "We need a Python developer with React experience"
        sample_resume_data = json.loads(extracted_data)
        
        # This might fail without OpenAI API key, but should not crash
        score_result = score_single_resume(sample_resume_data, sample_job_desc)
        
        if 'overall_score' in score_result:
            print(f"✅ Scoring completed: {score_result['overall_score']}/100")
        else:
            print("⚠️  Scoring completed with fallback result")
            
    except Exception as e:
        print(f"⚠️  Scoring failed (expected without OpenAI API): {e}")
    
    print("\n=== WORKFLOW TEST SUMMARY ===")
    print("✅ Core functionality is working!")
    print("✅ Import issues have been resolved!")
    print("✅ System is ready for use!")
    
    return True

# Run the comprehensive test
success = test_ai_service_workflow()

if success:
    print("\n🎉 ALL SYSTEMS GO!")
    print("Your AI recruitment system is ready to use!")
    print("\nNext steps:")
    print("1. Set OPENAI_API_KEY in your .env file for full AI functionality")
    print("2. Set DATABASE_URL for database integration")
    print("3. Run: python ai_service.py --file test_resume.txt")
else:
    print("\n❌ Some issues remain - check the error messages above")

## Summary

This notebook has helped you:

1. ✅ **Verify environment setup** - Check Python version, packages, and working directory
2. ✅ **Diagnose import errors** - Systematically test module imports
3. ✅ **Configure Python paths** - Ensure modules can be found
4. ✅ **Clean Python cache** - Remove stale .pyc files and __pycache__ directories
5. ✅ **Reload modules** - Refresh module definitions
6. ✅ **Test database connections** - Verify database module functionality
7. ✅ **Validate file paths** - Ensure all required files exist
8. ✅ **Handle errors comprehensively** - Test the complete AI workflow

## Next Steps

Your AI recruitment system should now be working! To use it:

1. **Set up environment variables** (in `.env` file):
   ```
   OPENAI_API_KEY=your_openai_api_key_here
   DATABASE_URL=your_database_url_here
   ```

2. **Test the AI service**:
   ```bash
   python ai_service.py --file test_resume.txt
   ```

3. **Use from your Next.js application**:
   - Upload resumes through the candidate portal
   - Apply for jobs to trigger AI scoring
   - View results in the HR dashboard

The system now has full AI-powered resume analysis capabilities! 🚀